# Extract VectorNet vector detections (raw DwC-A)

Downloads the **latest** VectorNet Darwin Core Archive from the GBIF IPT and **overwrites** the local raw copy under `data/raw/vector/`.

- Source: [VectorNet on IPT](https://ipt.gbif.org/resource?r=vndatabase) (`vndatabase`)
- GBIF UUID: `e497586a-1bdf-4f69-90eb-645d615762c8`
- License: CC-BY 4.0
- No GBIF account required

This notebook stops at raw validation (file list + occurrence smoke check). It does not clean or process the data.

In [1]:
from __future__ import annotations

import hashlib
import json
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

from tqdm import tqdm

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / "data" / "raw" / "vector"
ZIP_NAME = "vectornet_dwca.zip"
ZIP_PATH = RAW_DIR / ZIP_NAME
MANIFEST_PATH = RAW_DIR / "manifest.json"

SOURCE_URL = "https://ipt.gbif.org/archive.do?r=vndatabase"
RESOURCE = "vndatabase"
GBIF_DATASET_UUID = "e497586a-1bdf-4f69-90eb-645d615762c8"
CITATION = (
    "VectorNet, European Centre for Disease Prevention and Control, "
    "European Food Safety Authority. VectorNet. Training Organization. "
    "Occurrence dataset (CC-BY 4.0). "
    "https://ipt.gbif.org/resource?r=vndatabase"
)
DOWNLOAD_TIMEOUT_S = 300
CHUNK_SIZE = 1024 * 256

RAW_DIR, SOURCE_URL

(PosixPath('/workspaces/vector-guard/data/raw/vector'),
 'https://ipt.gbif.org/archive.do?r=vndatabase')

## Wipe local raw directory

Every run replaces the previous local copy with the archive fetched now.

In [2]:
def wipe_raw_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    for child in path.iterdir():
        if child.name == ".gitkeep":
            continue
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    (path / ".gitkeep").touch(exist_ok=True)


wipe_raw_dir(RAW_DIR)
print(f"Wiped and recreated: {RAW_DIR}")

Wiped and recreated: /workspaces/vector-guard/data/raw/vector


## Download latest DwC-A from IPT

In [3]:
def download_file(url: str, dest: Path, timeout_s: int = DOWNLOAD_TIMEOUT_S) -> int:
    request = Request(url, headers={"User-Agent": "vector-guard/0.1"})
    with urlopen(request, timeout=timeout_s) as response:
        total = response.headers.get("Content-Length")
        total_bytes = int(total) if total and total.isdigit() else None
        written = 0
        with dest.open("wb") as out, tqdm(
            total=total_bytes,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=dest.name,
        ) as bar:
            while True:
                chunk = response.read(CHUNK_SIZE)
                if not chunk:
                    break
                out.write(chunk)
                written += len(chunk)
                bar.update(len(chunk))
    return written


zip_bytes = download_file(SOURCE_URL, ZIP_PATH)
print(f"Downloaded {zip_bytes:,} bytes -> {ZIP_PATH}")

vectornet_dwca.zip: 29.0MB [00:11, 2.71MB/s]

Downloaded 30,443,144 bytes -> /workspaces/vector-guard/data/raw/vector/vectornet_dwca.zip


## Extract archive and write manifest

In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()


with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(RAW_DIR)

extracted = sorted(
    p.relative_to(RAW_DIR).as_posix()
    for p in RAW_DIR.rglob("*")
    if p.is_file() and p.name != ZIP_NAME
)

manifest = {
    "source_url": SOURCE_URL,
    "resource": RESOURCE,
    "gbif_dataset_uuid": GBIF_DATASET_UUID,
    "fetched_at": datetime.now(timezone.utc).isoformat(),
    "zip_name": ZIP_NAME,
    "zip_bytes": ZIP_PATH.stat().st_size,
    "sha256": sha256_file(ZIP_PATH),
    "citation": CITATION,
    "extracted_files": extracted,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Extracted {len(extracted)} files")
print(f"Wrote {MANIFEST_PATH}")
manifest

Extracted 4 files
Wrote /workspaces/vector-guard/data/raw/vector/manifest.json


{'source_url': 'https://ipt.gbif.org/archive.do?r=vndatabase',
 'resource': 'vndatabase',
 'gbif_dataset_uuid': 'e497586a-1bdf-4f69-90eb-645d615762c8',
 'fetched_at': '2026-11-10T14:36:47.471240+00:00',
 'zip_name': 'vectornet_dwca.zip',
 'zip_bytes': 30443144,
 'sha256': '5cff38259325a9b8d0a0b4226b42e763859780d4ec1d6d0f5a60958d2f2c1c79',
 'citation': 'VectorNet, European Centre for Disease Prevention and Control, European Food Safety Authority. VectorNet. Training Organization. Occurrence dataset (CC-BY 4.0). https://ipt.gbif.org/resource?r=vndatabase',
 'extracted_files': ['.gitkeep', 'eml.xml', 'meta.xml', 'occurrence.txt']}

## Smoke check

Load the core occurrence table via the shared `load_vector_detections` helper.

In [5]:
from vector_guard.data import load_vector_detections, resolve_vector_core_path

print("Files in raw dir:")
for rel in sorted(p.relative_to(RAW_DIR).as_posix() for p in RAW_DIR.rglob("*") if p.is_file()):
    print(f"  - {rel}")

core_path = resolve_vector_core_path(RAW_DIR)
df = load_vector_detections(RAW_DIR)
print(f"\nCore file: {core_path.name}")
print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}): {list(df.columns)[:12]}{'...' if len(df.columns) > 12 else ''}")
df.head(3)

Files in raw dir:
  - .gitkeep
  - eml.xml
  - manifest.json
  - meta.xml
  - occurrence.txt
  - vectornet_dwca.zip

Core file: occurrence.txt
Rows: 277,353
Columns (26): ['id', 'bibliographicCitation', 'basisOfRecord', 'dynamicProperties', 'occurrenceID', 'individualCount', 'sex', 'lifeStage', 'degreeOfEstablishment', 'associatedTaxa', 'eventID', 'eventDate']...


,id,bibliographicCitation,basisOfRecord,dynamicProperties,occurrenceID,individualCount,sex,lifeStage,degreeOfEstablishment,associatedTaxa,...,locationID,higherGeographyID,higherGeography,countryCode,locationRemarks,decimalLatitude,decimalLongitude,verbatimIdentification,identificationRemarks,scientificName
0,7E15E2D4-3F33-41A1-81DA-000025D68FE7,NaN,humanObservation,NaN,7E15E2D4-3F33-41A1-81DA-000025D68FE7,0,female,adult,Native - Present,host: Not applicable,...,54.07396 : 23.37122,PL843,Poland | Suwalski,PL,NaN,54.07396,23.37122,Culicoides newsteadi s.l.,morphological,Culicoides
1,4B85477C-5310-4E4E-8AA7-00004B42A332,NaN,humanObservation,NaN,4B85477C-5310-4E4E-8AA7-00004B42A332,25,NaN,egg,Invasive - Established,NaN,...,42.07000 : 12.67139,ITI43,Italy | Roma,IT,FALSE,42.07,12.671389,Aedes albopictus,morphological,Aedes albopictus
2,015BFCF3-1892-4F3A-AE1A-000058416ABE,NaN,humanObservation,NaN,015BFCF3-1892-4F3A-AE1A-000058416ABE,0,female,adult,Native - Obs. Absent,host: Not applicable,...,52.45000 : 16.85000,PL415,Poland | Miasto Pozna?,PL,NaN,52.45,16.85,Culicoides lupicaris,morphological,Culicoides lupicaris
